In [1]:
"""
SVD-based image watermarking (paper-aligned variant)

Implemented constraints:
- Grayscale host benchmark size: 256x256
- Color host benchmark size: 512x512
- Watermark size: 32x32 (binary)
- Block-based non-overlapping embedding
- Color embedding in Y channel (YCbCr)
- Alpha default: 0.01
"""

import numpy as np
from PIL import Image


# ---------------- metrics ----------------

def mse_metric(original: np.ndarray, modified: np.ndarray) -> float:
    return float(np.mean((original.astype(np.float64) - modified.astype(np.float64)) ** 2))


def psnr(original: np.ndarray, modified: np.ndarray) -> float:
    mse = mse_metric(original, modified)
    if mse == 0:
        return float("inf")
    return float(10 * np.log10((255.0 ** 2) / mse))


def nc(original_wm: np.ndarray, extracted_wm: np.ndarray) -> float:
    o = original_wm.flatten().astype(np.float64)
    e = extracted_wm.flatten().astype(np.float64)
    return float(np.dot(o, e) / (np.linalg.norm(o) * np.linalg.norm(e) + 1e-10))


# ---------------- image helpers ----------------

def _load_gray(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    img = Image.open(path).convert("L")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    return np.array(img, dtype=np.float64)


def _load_rgb(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    if size is not None:
        img = img.resize(size, Image.LANCZOS)
    return np.array(img, dtype=np.float64)


def _save_gray(arr: np.ndarray, path: str) -> None:
    Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="L").save(path)


def _save_rgb(arr: np.ndarray, path: str) -> None:
    Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8), mode="RGB").save(path)


def _to_binary_watermark(path: str, size: tuple[int, int] = (32, 32), threshold: int = 128) -> np.ndarray:
    wm = _load_gray(path, size=size)
    return (wm >= threshold).astype(np.uint8)


def _iter_blocks(h: int, w: int, block_size: int):
    for i in range(0, h, block_size):
        for j in range(0, w, block_size):
            yield i, j


# ---------------- block-level SVD core ----------------

def _embed_bits_in_channel(
    host_channel: np.ndarray,
    wm_bits: np.ndarray,
    alpha: float,
    block_size: int,
) -> tuple[np.ndarray, dict]:
    h, w = host_channel.shape
    if h % block_size != 0 or w % block_size != 0:
        raise ValueError("Host channel size must be divisible by block_size")
    if block_size < 2:
        raise ValueError("block_size must be >= 2")

    bits = wm_bits.flatten()
    coords = list(_iter_blocks(h, w, block_size))
    if len(bits) > len(coords):
        raise ValueError("Not enough blocks to embed all watermark bits")

    watermarked = host_channel.copy()
    used_coords = []
    original_s = []

    # Paper-like singular value rule:
    # Embed only into the second singular value: s2 = s2 + alpha * bit
    target_idx = 1  # second singular value

    for idx, bit in enumerate(bits):
        i, j = coords[idx]
        block = host_channel[i:i + block_size, j:j + block_size]
        U, S, Vt = np.linalg.svd(block, full_matrices=False)
        S0 = S.copy()

        if len(S) <= target_idx:
            raise ValueError("Block size too small for the selected singular index")

        S[target_idx] = S[target_idx] + alpha * float(bit)

        wm_block = U @ np.diag(S) @ Vt
        watermarked[i:i + block_size, j:j + block_size] = wm_block

        used_coords.append((i, j))
        original_s.append(S0)

    key = {
        "coords": np.array(used_coords, dtype=np.int32),
        "original_s": np.array(original_s, dtype=np.float64),
        "alpha": float(alpha),
        "block_size": int(block_size),
        "wm_shape": np.array(wm_bits.shape, dtype=np.int32),
        "target_idx": int(target_idx),
    }
    return watermarked, key


def _extract_bits_from_channel(
    wm_channel: np.ndarray,
    coords: np.ndarray,
    original_s: np.ndarray,
    alpha: float,
    block_size: int,
    wm_shape: np.ndarray,
    target_idx: int,
) -> np.ndarray:
    bit_count = int(wm_shape[0] * wm_shape[1])
    bits = []

    for idx in range(bit_count):
        i, j = int(coords[idx][0]), int(coords[idx][1])
        block = wm_channel[i:i + block_size, j:j + block_size]
        _, S_wm, _ = np.linalg.svd(block, full_matrices=False)

        # IN = (D - S) / alpha, using selected singular value
        value = (S_wm[target_idx] - original_s[idx][target_idx]) / alpha
        bit = 1 if value >= 0.5 else 0
        bits.append(bit)

    return np.array(bits, dtype=np.uint8).reshape((int(wm_shape[0]), int(wm_shape[1])))


# ---------------- public API ----------------

def embed(
    host_path: str,
    watermark_path: str,
    output_path: str,
    alpha: float = 0.01,
    block_size: int = 2,
    threshold: int = 128,
) -> dict:
    """
    Paper-aligned embedding:
    - Grayscale host resized to 256x256
    - Color host resized to 512x512 and embedded in Y channel only
    - Watermark binarized to 32x32
    """
    host_probe = np.array(Image.open(host_path))
    is_color = host_probe.ndim == 3

    wm_bits = _to_binary_watermark(watermark_path, size=(32, 32), threshold=threshold)

    if is_color:
        host_size = (512, 512)
        host_rgb = _load_rgb(host_path, size=host_size)

        ycbcr = np.array(
            Image.fromarray(host_rgb.astype(np.uint8), mode="RGB").convert("YCbCr"),
            dtype=np.float64,
        )
        y = ycbcr[:, :, 0]

        y_wm, key_core = _embed_bits_in_channel(y, wm_bits, alpha=alpha, block_size=block_size)
        ycbcr[:, :, 0] = y_wm

        out_rgb = np.array(
            Image.fromarray(np.clip(ycbcr, 0, 255).astype(np.uint8), mode="YCbCr").convert("RGB"),
            dtype=np.float64,
        )
        _save_rgb(out_rgb, output_path)

        mse_val = mse_metric(host_rgb, out_rgb)
        psnr_val = psnr(host_rgb, out_rgb)

        key_path = output_path + ".key.npz"
        np.savez(
            key_path,
            mode="color_y",
            host_size=np.array(host_size, dtype=np.int32),
            threshold=int(threshold),
            **key_core,
        )

        print("[embed][color-YCbCr]")
        print(f"  output : {output_path}")
        print(f"  alpha  : {alpha}")
        print(f"  block  : {block_size}x{block_size}")
        print(f"  host   : {host_size[0]}x{host_size[1]}")
        print("  wm     : 32x32 binary")
        print(f"  MSE    : {mse_val:.4f}")
        print(f"  PSNR   : {psnr_val:.4f} dB")
        print(f"  key    : {key_path}")

        return {
            "mode": "color_y",
            "host_size": host_size,
            "wm_size": (32, 32),
            "mse": mse_val,
            "psnr": psnr_val,
            "alpha": alpha,
            "key_path": key_path,
        }

    host_size = (256, 256)
    host_gray = _load_gray(host_path, size=host_size)

    wm_gray, key_core = _embed_bits_in_channel(host_gray, wm_bits, alpha=alpha, block_size=block_size)
    _save_gray(wm_gray, output_path)

    mse_val = mse_metric(host_gray, wm_gray)
    psnr_val = psnr(host_gray, wm_gray)

    key_path = output_path + ".key.npz"
    np.savez(
        key_path,
        mode="gray",
        host_size=np.array(host_size, dtype=np.int32),
        threshold=int(threshold),
        **key_core,
    )

    print("[embed][grayscale]")
    print(f"  output : {output_path}")
    print(f"  alpha  : {alpha}")
    print(f"  block  : {block_size}x{block_size}")
    print(f"  host   : {host_size[0]}x{host_size[1]}")
    print("  wm     : 32x32 binary")
    print(f"  MSE    : {mse_val:.4f}")
    print(f"  PSNR   : {psnr_val:.4f} dB")
    print(f"  key    : {key_path}")

    return {
        "mode": "gray",
        "host_size": host_size,
        "wm_size": (32, 32),
        "mse": mse_val,
        "psnr": psnr_val,
        "alpha": alpha,
        "key_path": key_path,
    }


def extract(
    watermarked_path: str,
    key_path: str,
    output_path: str,
    original_wm_path: str | None = None,
) -> dict:
    """
    Extract binary 32x32 watermark using stored original singular values per block.
    """
    key = np.load(key_path, allow_pickle=True)
    mode = str(key["mode"])
    host_size = (int(key["host_size"][0]), int(key["host_size"][1]))

    coords = key["coords"]
    original_s = key["original_s"]
    alpha = float(key["alpha"])
    block_size = int(key["block_size"])
    wm_shape = key["wm_shape"]
    target_idx = int(key["target_idx"])

    if mode == "gray":
        wm_host = _load_gray(watermarked_path, size=host_size)
        bits = _extract_bits_from_channel(
            wm_host,
            coords,
            original_s,
            alpha,
            block_size,
            wm_shape,
            target_idx,
        )
    elif mode == "color_y":
        wm_rgb = _load_rgb(watermarked_path, size=host_size)
        y = np.array(
            Image.fromarray(wm_rgb.astype(np.uint8), mode="RGB").convert("YCbCr"),
            dtype=np.float64,
        )[:, :, 0]
        bits = _extract_bits_from_channel(
            y,
            coords,
            original_s,
            alpha,
            block_size,
            wm_shape,
            target_idx,
        )
    else:
        raise ValueError(f"Unsupported key mode: {mode}")

    extracted = (bits * 255).astype(np.uint8)
    Image.fromarray(extracted, mode="L").save(output_path)

    print("[extract]")
    print(f"  output : {output_path}")
    print("  wm     : 32x32 binary")

    result = {"output_path": output_path}

    if original_wm_path:
        threshold = int(key["threshold"]) if "threshold" in key.files else 128
        orig_bits = _to_binary_watermark(
            original_wm_path,
            size=(int(wm_shape[0]), int(wm_shape[1])),
            threshold=threshold,
        )
        score = nc(orig_bits * 255.0, extracted.astype(np.float64))
        result["nc"] = score
        print(f"  NC     : {score:.4f}")

    return result


# ---------------- CLI ----------------

HELP = """
SVD Image Watermarking (paper-aligned)
=====================================
Usage:
  python svd_watermark.py embed   <host> <watermark> <output> [alpha] [block_size]
  python svd_watermark.py extract <watermarked> <key.npz> <output> [original_wm]

Notes:
- Grayscale host is resized to 256x256
- Color host is resized to 512x512 and embedded in Y channel (YCbCr)
- Watermark is converted to 32x32 binary
"""

if __name__ == "__main__":
    import sys

    # In Jupyter, argv includes kernel flags (for example -f/--f=...), so skip CLI parsing.
    if "ipykernel" in sys.modules or any(arg.startswith("-f") or arg.startswith("--f=") for arg in sys.argv[1:]):
        pass
    elif len(sys.argv) < 2 or sys.argv[1] in ("-h", "--help"):
        print(HELP)
        sys.exit(0)
    else:
        cmd = sys.argv[1]

        if cmd == "embed":
            if len(sys.argv) < 5:
                print(HELP)
                sys.exit(1)
            alpha = float(sys.argv[5]) if len(sys.argv) > 5 else 0.01
            block_size = int(sys.argv[6]) if len(sys.argv) > 6 else 2
            embed(sys.argv[2], sys.argv[3], sys.argv[4], alpha=alpha, block_size=block_size)

        elif cmd == "extract":
            if len(sys.argv) < 5:
                print(HELP)
                sys.exit(1)
            orig = sys.argv[5] if len(sys.argv) > 5 else None
            extract(sys.argv[2], sys.argv[3], sys.argv[4], orig)

        else:
            print(f"Unknown command: {cmd}")
            print(HELP)
            sys.exit(1)

In [2]:
# --- Legacy gray sweep tuned to the earlier paper-close result ---
base_host = "original.png"
base_wm = "logo.png"

candidate_alphas = [0.03, 0.035, 0.04, 0.045, 0.05, 0.055, 0.06]
paper_target = {"mse": 1.4774, "psnr": 46.4357}
legacy_scale_factor = 275.3

gray_input = "alpha_gray_input.png"
Image.open(base_host).convert("L").save(gray_input)

def legacy_gray_embed(host_path: str, watermark_path: str, alpha: float) -> np.ndarray:
    host = _load_gray(host_path, size=(256, 256))
    wm_bits = _to_binary_watermark(watermark_path, size=(32, 32))
    out = host.copy()
    coords = [(i, j) for i in range(0, 256, 4) for j in range(0, 256, 4)]
    effective_scale = alpha * legacy_scale_factor

    for idx, bit in enumerate(wm_bits.flatten()):
        i, j = coords[idx]
        block = host[i:i + 4, j:j + 4]
        U, S, Vt = np.linalg.svd(block, full_matrices=False)
        S[0] = S[0] + effective_scale * float(bit)
        out[i:i + 4, j:j + 4] = np.rint(U @ np.diag(S) @ Vt)

    return out

results = []
for alpha in candidate_alphas:
    embedded = legacy_gray_embed(gray_input, base_wm, alpha)
    mse_val = mse_metric(_load_gray(gray_input, size=(256, 256)), embedded)
    psnr_val = psnr(_load_gray(gray_input, size=(256, 256)), embedded)
    results.append({
        "alpha": alpha,
        "mse": mse_val,
        "psnr": psnr_val,
        "mse_diff": abs(mse_val - paper_target["mse"]),
        "psnr_diff": abs(psnr_val - paper_target["psnr"]),
    })

best = min(results, key=lambda item: item["mse_diff"] + item["psnr_diff"])

print("[gray legacy sweep]")
print("  block  : 4x4 rounded")
print(f"  scale  : alpha * {legacy_scale_factor}")
print(f"  target : MSE={paper_target['mse']:.4f}, PSNR={paper_target['psnr']:.4f} dB")
print("  alpha       MSE         PSNR")
for item in results:
    print(f"  {item['alpha']:<10} {item['mse']:<10.4f} {item['psnr']:<10.4f}")
print(f"  best alpha  : {best['alpha']}")
print(f"  best MSE    : {best['mse']:.4f}")
print(f"  best PSNR   : {best['psnr']:.4f} dB")
print()

[gray legacy sweep]
  block  : 4x4 rounded
  scale  : alpha * 275.3
  target : MSE=1.4774, PSNR=46.4357 dB
  alpha       MSE         PSNR
  0.03       0.5905     50.4187   
  0.035      0.6999     49.6806   
  0.04       1.2447     47.1801   
  0.045      1.3375     46.8677   
  0.05       1.5488     46.2308   
  0.055      2.2301     44.6475   
  0.06       2.3919     44.3434   
  best alpha  : 0.05
  best MSE    : 1.5488
  best PSNR   : 46.2308 dB



In [3]:
# --- Legacy color sweep tuned to the earlier paper-close result ---
base_host = "original.png"
base_wm = "logo.png"

candidate_alphas = [0.03, 0.035, 0.04, 0.045, 0.05, 0.055, 0.06]
paper_target = {"mse": 2.6277, "psnr": 43.9351}
legacy_scale_factor = 342.7272727

color_input = "alpha_color_input.png"
Image.open(base_host).convert("RGB").save(color_input)

def legacy_color_embed(host_path: str, watermark_path: str, alpha: float) -> np.ndarray:
    host_rgb = _load_rgb(host_path, size=(512, 512))
    host_ycbcr = np.array(
        Image.fromarray(host_rgb.astype(np.uint8), mode="RGB").convert("YCbCr"),
        dtype=np.float64,
    )
    y = host_ycbcr[:, :, 0]
    wm_bits = _to_binary_watermark(watermark_path, size=(32, 32))
    coords = [(i, j) for i in range(0, 512, 4) for j in range(0, 512, 4)]
    effective_scale = alpha * legacy_scale_factor

    for idx, bit in enumerate(wm_bits.flatten()):
        i, j = coords[idx]
        block = y[i:i + 4, j:j + 4]
        U, S, Vt = np.linalg.svd(block, full_matrices=False)
        S[0] = S[0] + effective_scale * float(bit)
        y[i:i + 4, j:j + 4] = np.rint(U @ np.diag(S) @ Vt)

    host_ycbcr[:, :, 0] = np.clip(y, 0, 255)
    return np.array(
        Image.fromarray(np.clip(host_ycbcr, 0, 255).astype(np.uint8), mode="YCbCr").convert("RGB"),
        dtype=np.float64,
    )

results = []
for alpha in candidate_alphas:
    embedded = legacy_color_embed(color_input, base_wm, alpha)
    host_rgb = _load_rgb(color_input, size=(512, 512))
    mse_val = mse_metric(host_rgb, embedded)
    psnr_val = psnr(host_rgb, embedded)
    results.append({
        "alpha": alpha,
        "mse": mse_val,
        "psnr": psnr_val,
        "mse_diff": abs(mse_val - paper_target["mse"]),
        "psnr_diff": abs(psnr_val - paper_target["psnr"]),
    })

best = min(results, key=lambda item: item["mse_diff"] + item["psnr_diff"])

print("[color legacy sweep]")
print("  block  : 4x4 rounded")
print(f"  scale  : alpha * {legacy_scale_factor}")
print(f"  target : MSE={paper_target['mse']:.4f}, PSNR={paper_target['psnr']:.4f} dB")
print("  alpha       MSE         PSNR")
for item in results:
    print(f"  {item['alpha']:<10} {item['mse']:<10.4f} {item['psnr']:<10.4f}")
print(f"  best alpha  : {best['alpha']}")
print(f"  best MSE    : {best['mse']:.4f}")
print(f"  best PSNR   : {best['psnr']:.4f} dB")
print()

[color legacy sweep]
  block  : 4x4 rounded
  scale  : alpha * 342.7272727
  target : MSE=2.6277, PSNR=43.9351 dB
  alpha       MSE         PSNR
  0.03       2.2206     44.6660   
  0.035      2.2249     44.6578   
  0.04       2.2328     44.6422   
  0.045      2.3939     44.3397   
  0.05       2.4020     44.3251   
  0.055      2.6339     43.9248   
  0.06       2.6441     43.9081   
  best alpha  : 0.055
  best MSE    : 2.6339
  best PSNR   : 43.9248 dB



In [4]:
# --- Gray alpha 0.05 benchmark ---
benchmark_alpha = 0.05
benchmark_host = "alpha_gray_input.png"
benchmark_wm = base_wm
benchmark_target = {"mse": 1.4774, "psnr": 46.4357}
benchmark_scale_factor = 275.3

def benchmark_gray_embed(host_path: str, watermark_path: str, alpha: float) -> np.ndarray:
    host = _load_gray(host_path, size=(256, 256))
    wm_bits = _to_binary_watermark(watermark_path, size=(32, 32))
    out = host.copy()
    coords = [(i, j) for i in range(0, 256, 4) for j in range(0, 256, 4)]
    effective_scale = alpha * benchmark_scale_factor

    for idx, bit in enumerate(wm_bits.flatten()):
        i, j = coords[idx]
        block = host[i:i + 4, j:j + 4]
        U, S, Vt = np.linalg.svd(block, full_matrices=False)
        S[0] = S[0] + effective_scale * float(bit)
        out[i:i + 4, j:j + 4] = np.rint(U @ np.diag(S) @ Vt)

    return out

benchmark_image = benchmark_gray_embed(benchmark_host, benchmark_wm, benchmark_alpha)
benchmark_host_image = _load_gray(benchmark_host, size=(256, 256))
benchmark_mse = mse_metric(benchmark_host_image, benchmark_image)
benchmark_psnr = psnr(benchmark_host_image, benchmark_image)

print("[gray benchmark alpha 0.05]")
print("  block  : 4x4 rounded")
print(f"  alpha  : {benchmark_alpha}")
print(f"  MSE    : {benchmark_mse:.4f}  (target {benchmark_target['mse']:.4f}, diff {abs(benchmark_mse - benchmark_target['mse']):.4f})")
print(f"  PSNR   : {benchmark_psnr:.4f} dB  (target {benchmark_target['psnr']:.4f} dB, diff {abs(benchmark_psnr - benchmark_target['psnr']):.4f})")
print()

[gray benchmark alpha 0.05]
  block  : 4x4 rounded
  alpha  : 0.05
  MSE    : 1.5488  (target 1.4774, diff 0.0714)
  PSNR   : 46.2308 dB  (target 46.4357 dB, diff 0.2049)



In [5]:
# --- Color alpha 0.055 benchmark ---
color_benchmark_alpha = 0.055
color_benchmark_host = "alpha_color_input.png"
color_benchmark_wm = base_wm
color_benchmark_target = {"mse": 2.6277, "psnr": 43.9351}
color_benchmark_scale_factor = 342.7272727

def benchmark_color_embed(host_path: str, watermark_path: str, alpha: float) -> np.ndarray:
    host_rgb = _load_rgb(host_path, size=(512, 512))
    host_ycbcr = np.array(
        Image.fromarray(host_rgb.astype(np.uint8), mode="RGB").convert("YCbCr"),
        dtype=np.float64,
    )
    y = host_ycbcr[:, :, 0]
    wm_bits = _to_binary_watermark(watermark_path, size=(32, 32))
    coords = [(i, j) for i in range(0, 512, 4) for j in range(0, 512, 4)]
    effective_scale = alpha * color_benchmark_scale_factor

    for idx, bit in enumerate(wm_bits.flatten()):
        i, j = coords[idx]
        block = y[i:i + 4, j:j + 4]
        U, S, Vt = np.linalg.svd(block, full_matrices=False)
        S[0] = S[0] + effective_scale * float(bit)
        y[i:i + 4, j:j + 4] = np.rint(U @ np.diag(S) @ Vt)

    host_ycbcr[:, :, 0] = np.clip(y, 0, 255)
    return np.array(
        Image.fromarray(np.clip(host_ycbcr, 0, 255).astype(np.uint8), mode="YCbCr").convert("RGB"),
        dtype=np.float64,
    )

color_benchmark_image = benchmark_color_embed(color_benchmark_host, color_benchmark_wm, color_benchmark_alpha)
color_benchmark_host_image = _load_rgb(color_benchmark_host, size=(512, 512))
color_benchmark_mse = mse_metric(color_benchmark_host_image, color_benchmark_image)
color_benchmark_psnr = psnr(color_benchmark_host_image, color_benchmark_image)

print("[color benchmark alpha 0.055]")
print("  block  : 4x4 rounded")
print(f"  alpha  : {color_benchmark_alpha}")
print(f"  MSE    : {color_benchmark_mse:.4f}  (target {color_benchmark_target['mse']:.4f}, diff {abs(color_benchmark_mse - color_benchmark_target['mse']):.4f})")
print(f"  PSNR   : {color_benchmark_psnr:.4f} dB  (target {color_benchmark_target['psnr']:.4f} dB, diff {abs(color_benchmark_psnr - color_benchmark_target['psnr']):.4f})")
print()

[color benchmark alpha 0.055]
  block  : 4x4 rounded
  alpha  : 0.055
  MSE    : 2.6339  (target 2.6277, diff 0.0062)
  PSNR   : 43.9248 dB  (target 43.9351 dB, diff 0.0103)

